In [ ]:
pip install shap

In [ ]:
import joblib
import shap
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

In [ ]:
best_model = joblib.load("best_model.pkl")

tfidf = joblib.load("tfidf_vectorizer.pkl")

X_train = joblib.load("X_train.pkl")
X_test = joblib.load("X_test.pkl")

y_train = joblib.load("y_train.pkl")
y_test = joblib.load("y_test.pkl")

In [ ]:
tfidf_features = tfidf.get_feature_names_out()

manual_features = [

    "char_count",
    "word_count",
    "sentence_count",
    "avg_word_length",
    "digit_ratio",
    "special_ratio",
    "exclamation_count",
    "question_count",

    "has_email",
    "gmail_email",
    "yahoo_email",
    "outlook_email",

    "salary_present",
    "salary_daily",
    "salary_monthly",
    "salary_hourly",

    "avg_sentence_length",
    "unique_word_ratio",
    "repetition_score",
    "long_word_ratio",
    "numeric_token_ratio"

]

feature_names = list(tfidf_features) + manual_features

print(len(feature_names))

In [ ]:
explainer = shap.TreeExplainer(best_model)

In [ ]:
sample = explain_prediction(job_description)
shap_values = explainer.shap_values(sample)
shap_df = pd.DataFrame({
    "feature": feature_names,
    "importance": shap_values[0]
})

shap_df = shap_df.sort_values(
    by="importance",
    key=np.abs,
    ascending=False
)

shap_df.head(20)

In [ ]:
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=sample.toarray()[0],
        feature_names=feature_names
    )
)

In [ ]:

sample_index = 0      # Change this to explain any row
sample = X_test[sample_index]
sample_true_label = y_test.iloc[sample_index]
sample_prediction = best_model.predict(sample)[0]
sample_probability = best_model.predict_proba(sample)[0][1]
shap_values = explainer.shap_values(sample)

shap_df = pd.DataFrame({
    "Feature": feature_names,
    "SHAP_Value": shap_values[0]
})

shap_df["Abs"] = shap_df["SHAP_Value"].abs()
shap_df = shap_df.sort_values(
    by="Abs",
    ascending=False
)

print("="*60)
print("True Label :", sample_true_label)
print("Prediction :", sample_prediction)
print("Fake Probability :", round(sample_probability*100,2),"%")
print("="*60)

shap_df.head(20)